# 02 - Data Cleaning and Data Quality Report
**Project:** Air Quality & Pollution Intelligence - Data Mining and Business Intelligence

**Objective:** Remove duplicates, fix types, handle impossible and missing values, flag (not delete) extreme values, and document every decision with the number of rows it touched.

**How to read this notebook:** every number printed below is produced by the
code in this notebook from `data/raw/Air_quality_data.csv`. Column names are
discovered at runtime through `src/config.py`, so nothing is assumed.


In [1]:
"""Environment bootstrap: make src/ importable and pin the working directory."""
import sys, os, warnings
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.chdir(PROJECT_ROOT)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 90)
%matplotlib inline
print("project root:", PROJECT_ROOT)

project root: C:\Users\DELL\OneDrive\Desktop\Air Quality


In [2]:
import config as C
import data_utils as U
import preprocessing as P
import statistics_analysis as S

raw = U.load_raw()
before = U.summary(raw)
print("before cleaning:", before["Total Rows"], "rows,",
      before["Total Missing Values"], "missing cells,",
      before["Duplicate Rows"], "duplicate rows")

before cleaning: 18265 rows, 0 missing cells, 0 duplicate rows


## 1. Cleaning strategy - and why each rule was chosen

| Problem | Rule applied | Why this rule |
|---|---|---|
| Exact duplicate rows | `drop_duplicates()` | Re-imported rows would double-count every average |
| Text in a numeric column | `to_numeric(errors="coerce")` | An unreadable value is not a measurement |
| Negative concentration | set to missing, then imputed | Physically impossible, so it must be an error |
| Missing numeric value | median of the **same city**, global median as fallback | Pollutant distributions are skewed and each city has its own baseline; a mean would be dragged by extreme (possibly genuine) values |
| Missing category | explicit `Unknown` label | Inventing a city or an AQI band would fabricate evidence |
| Extreme value | **flagged, kept** | A high PM day can be a real pollution episode; deleting it removes the events a dashboard exists to show |

Every row of this table is executed by `src/preprocessing.py`, and the notebook
prints how many values each rule actually touched.

In [3]:
clean, log, before, after = P.clean(raw)
print("rows before:", before["Total Rows"], "| rows after:", after["Total Rows"])
print("columns before:", before["Total Columns"],
      "| columns after:", after["Total Columns"],
      "(extra columns are the documented imputation/outlier flags)")

rows before: 18265 | rows after: 18265
columns before: 13 | columns after: 22 (extra columns are the documented imputation/outlier flags)


## 2. Cleaning audit log

In [4]:
from IPython.display import display
display(log)
log.to_csv(C.PROCESSED_DIR / "cleaning_log.csv", index=False)

,Step,Action Performed,Method / Rule,Rows / Values Affected,Justification
0,1,Duplicate removal,drop_duplicates() on the full record,0,"Checked for exact duplicates; none existed, so no rows were dropped."
1,2,Uniqueness check,"duplicated(City, Datetime)",0,"Verified: one record per city per day, so the panel is balanced."
2,3,Numeric coercion check,'PM2.5' already fully numeric,0,Verified rather than assumed; nothing had to be converted.
3,4,Numeric coercion check,'PM10' already fully numeric,0,Verified rather than assumed; nothing had to be converted.
4,5,Numeric coercion check,'NO' already fully numeric,0,Verified rather than assumed; nothing had to be converted.
5,6,Numeric coercion check,'NO2' already fully numeric,0,Verified rather than assumed; nothing had to be converted.
6,7,Numeric coercion check,'NOx' already fully numeric,0,Verified rather than assumed; nothing had to be converted.
7,8,Numeric coercion check,'NH3' already fully numeric,0,Verified rather than assumed; nothing had to be converted.
8,9,Numeric coercion check,'CO' already fully numeric,0,Verified rather than assumed; nothing had to be converted.
9,10,Numeric coercion check,'SO2' already fully numeric,0,Verified rather than assumed; nothing had to be converted.


### Reading the log honestly

On this particular file most rules report `0` affected values. That is a real
finding, not a failure: the supplied data contains no duplicates, no missing
cells and no negative concentrations. The rules are still executed so that the
same pipeline is safe on any other extract of this dataset.

In [5]:
zeroes = int((log["Rows / Values Affected"] == 0).sum())
print(f"{zeroes} of {len(log)} cleaning steps affected zero values in this file")
print("steps that did change something:")
display(log[log["Rows / Values Affected"] > 0])

42 of 43 cleaning steps affected zero values in this file
steps that did change something:


,Step,Action Performed,Method / Rule,Rows / Values Affected,Justification
33,34,Ordinal encoding,"'AQI_Bucket' cast to ordered category ['Severe', 'Very Poor', 'Poor', 'Moderate', 'Sat...",6,"AQI bands have a natural severity order, so models and charts sort them meaningfully i..."


## 3. Validity checks on the measurements

In [6]:
polls = C.pollutant_columns(clean.columns)
checks = []
for col in polls + [C.AQI_COL]:
    s = clean[col]
    checks.append({"Column": col, "Negative": int((s < 0).sum()), "Zero": int((s == 0).sum()),
                   "Min": s.min(), "Max": s.max(),
                   "Distinct values": int(s.nunique()),
                   "Values above observed max-1 step": int((s > s.max() - 0.1).sum())})
checks = pd.DataFrame(checks)
display(checks)

,Column,Negative,Zero,Min,Max,Distinct values,Values above observed max-1 step
0,PM2.5,0,1,0.0,499.9,4876,8
1,PM10,0,2,0.0,600.0,5709,2
2,NO,0,5,0.0,200.0,2000,4
3,NO2,0,7,0.0,150.0,1501,7
4,NOx,0,1,0.0,250.0,2498,6
5,NH3,0,13,0.0,50.0,501,16
6,CO,0,11,0.0,10.0,1001,174
7,SO2,0,10,0.0,100.0,1001,14
8,O3,0,6,0.0,200.0,2001,5
9,AQI,0,0,26.9,500.0,3848,6


### What the range boundaries tell us

Each pollutant stops at a round maximum (PM2.5 at 499.9, PM10 at 600, NO at 200,
NH3 at 50, CO at 10 ...) and occupies a complete one-decimal grid (for example
NO has exactly 2,000 distinct values on 0-200). Real monitoring stations report
instrument noise and detection limits, not perfect grids clipped at round
numbers. This is recorded here as a data-provenance observation and tested
statistically in notebook 05.

In [7]:
grid = pd.DataFrame({
    "Column": polls,
    "Distinct": [int(clean[p].nunique()) for p in polls],
    "Possible 1-decimal steps on [min,max]": [
        int(round((clean[p].max() - clean[p].min()) * 10)) + 1 for p in polls],
})
grid["Grid filled (%)"] = (100 * grid["Distinct"] / grid["Possible 1-decimal steps on [min,max]"]).round(1)
grid

,Column,Distinct,"Possible 1-decimal steps on [min,max]",Grid filled (%)
0,PM2.5,4876,5000,97.5
1,PM10,5709,6001,95.1
2,NO,2000,2001,100.0
3,NO2,1501,1501,100.0
4,NOx,2498,2501,99.9
5,NH3,501,501,100.0
6,CO,1001,101,991.1
7,SO2,1001,1001,100.0
8,O3,2001,2001,100.0


## 4. Extreme values

The IQR rule flags records outside Q1-1.5*IQR and Q3+1.5*IQR. The flags are kept
as columns; no row is deleted.

In [8]:
outlier_cols = [c for c in clean.columns if c.endswith("_Outlier")]
counts = clean[outlier_cols].sum()
print("records flagged as extreme, by pollutant:")
display(counts.rename("flagged").to_frame().assign(
    percent=(100 * counts / len(clean)).round(3)))
print()
print("Any record flagged on any pollutant:",
      int((clean[outlier_cols].sum(axis=1) > 0).sum()))

records flagged as extreme, by pollutant:


,flagged,percent
PM2.5_Outlier,0,0.0
PM10_Outlier,0,0.0
NO_Outlier,0,0.0
NO2_Outlier,0,0.0
NOx_Outlier,0,0.0
NH3_Outlier,0,0.0
CO_Outlier,0,0.0
SO2_Outlier,0,0.0
O3_Outlier,0,0.0



Any record flagged on any pollutant: 0


## 5. AQI band vs AQI value - internal consistency

`config.py` carries one AQI band table (the CPCB NAAQI ranges: 0-50 Good,
51-100 Satisfactory, 101-200 Moderate, 201-300 Poor, 301-400 Very Poor,
401-500 Severe). The rest of the project maps AQI onto a band through
`U.band_from_aqi`, so if that table were wrong the error would show up here and
nowhere else. Two questions are answered with the data in front of us:

1. does the supplied `AQI_Bucket` agree with the band table, and
2. does every band label sit on a contiguous, non-overlapping stretch of the
   AQI number line (a band that overlapped its neighbour would mean the scale is
   not the one the labels were built from).

This is also the leakage warning used later in notebook 08.

In [9]:
band_check = clean[[C.AQI_COL, C.TARGET_COL]].copy()
band_check["Expected"] = U.band_from_aqi(clean[C.AQI_COL])
agree = float((band_check["Expected"].astype(str)
               == band_check[C.TARGET_COL].astype(str)).mean())
print(f"{100*agree:.2f}% of rows have a bucket consistent with the band table in config.py")
print()
print("Cross-tabulation of supplied label against recomputed label (a clean "
      "diagonal means the two are the same scale):")
display(pd.crosstab(band_check[C.TARGET_COL], band_check["Expected"]))
print()
print("Observed AQI range per label, straight from the data:")
display(band_check.groupby(C.TARGET_COL, observed=True)[C.AQI_COL]
        .agg(["min", "max", "count"])
        .reindex([b for b in C.AQI_BUCKET_ORDER if b in set(band_check[C.TARGET_COL])]))
print()
print("Conclusion: AQI_Bucket is a deterministic function of AQI on this scale.")
print("Consequence: AQI must NOT be used to predict AQI_Bucket (data leakage).")

100.00% of rows have a bucket consistent with the band table in config.py

Cross-tabulation of supplied label against recomputed label (a clean diagonal means the two are the same scale):


Expected,Good,Moderate,Poor,Satisfactory,Severe,Very Poor
AQI_Bucket,,,,,,
Severe,0,0,0,0,5258,0
Very Poor,0,0,0,0,0,5501
Poor,0,0,3129,0,0,0
Moderate,0,4258,0,0,0,0
Satisfactory,0,0,0,113,0,0
Good,6,0,0,0,0,0



Observed AQI range per label, straight from the data:


,min,max,count
AQI_Bucket,,,
Severe,401.0,500.0,5258
Very Poor,301.0,400.0,5501
Poor,201.0,300.0,3129
Moderate,101.0,200.0,4258
Satisfactory,51.0,99.9,113
Good,26.9,49.2,6



Conclusion: AQI_Bucket is a deterministic function of AQI on this scale.
Consequence: AQI must NOT be used to predict AQI_Bucket (data leakage).


## 6. Before / after data quality report

In [10]:
comparison = U.quality_comparison(before, after)
display(comparison)
comparison.to_csv(C.DATA_QUALITY_CSV, index=False)
clean.to_csv(C.PROCESSED_DIR / "cleaned_stage02.csv", index=False)
print("saved: data/processed/data_quality_report.csv, cleaned_stage02.csv")

,Metric,Before Cleaning,After Cleaning
0,Total Rows,18265,18265
1,Total Columns,13,22
2,Duplicate Rows,0,0
3,Total Missing Values,0,0
4,Columns With Missing,0,0
5,Unique Cities,5,5
6,Date Start,2015-01-01,2015-01-01
7,Date End,2024-12-31,2024-12-31
8,Numerical Columns,10,19
9,Categorical Columns,3,3


saved: data/processed/data_quality_report.csv, cleaned_stage02.csv


**Note on the column count:** "After cleaning" is larger because the
documented flag columns (`*_Outlier`, `Imputed_*`) were added. No measurement
column was removed.

## Verification checklist

- [x] Duplicates counted before and after
- [x] Date parsed to datetime
- [x] Numeric and categorical columns identified from the data
- [x] Missing values handled with a stated rule (or verified absent)
- [x] Impossible values checked
- [x] Extreme values flagged, not deleted, with justification
- [x] Internal consistency of AQI and AQI_Bucket tested
- [x] Before/after report saved

**Common errors**

| Error | Meaning | Fix |
|---|---|---|
| `SettingWithCopyWarning` | assigning through a slice | the module always works on `df.copy()` and uses `.loc` |
| `InvalidComparison` on AQI_Bucket | column is an ordered category | compare with `.astype(str)` |

**Next:** `03_feature_engineering.ipynb`.